## Data Preparation du projet African Crop Yield Intelligence

Ce notebook applique les corrections identifiées lors du Data Understanding et produit les jeux de données préparés. Il fait suite à `02_data_understanding.ipynb` et à la décision de passer en Data Preparation sous trois conditions.

**Périmètre**

- Les fichiers `regional`, `survey` et `cropcut` sont préparés séparément, conformément à la première condition de la décision.
- Le fichier `point` est exclu, car il ne porte aucune des cibles retenues.

**Cibles retenues**

- `yieldTonHa` de `regional` aux niveaux administratifs 1 et 2, pour la généralisation spatiale et temporelle.
- `YieldUsingGPS_ton_ha_` de `survey`, à l'échelle de la parcelle.
- `Yield_ton_ha_` de `cropcut`, comme référence de validation pour l'Éthiopie.

**Méthode**

Toutes les corrections sont implémentées sous forme de fonctions testées dans `src/data/`, ce qui satisfait la deuxième condition de la décision. Ce notebook se limite à les exécuter, à contrôler leurs effets et à exporter les résultats. Les fichiers bruts ne sont jamais modifiés. Chaque correction ajoute un indicateur préfixé par `flag_`, de sorte que toute ligne modifiée ou écartée reste traçable.

In [1]:
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from src.data.load import DOSSIER_TRAITE, charger_brut
from src.data.prepare import exporter, preparer_tout

pd.set_option("display.max_columns", 40)

### 1. Tests unitaires

La suite de tests vérifie chaque fonction de préparation sur des cas construits pour l'occasion avant toute exécution sur les données réelles.

In [2]:
execution = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-t", "."],
    cwd=RACINE, capture_output=True, text=True,
)
print(execution.stderr.strip().splitlines()[-3:])

['Ran 53 tests in 0.260s', '', 'OK']


### 2. Exécution du pipeline

Les fichiers bruts sont chargés depuis le cache de `data/interim`, indexé par l'empreinte SHA-256 de chaque fichier source. Une copie de chaque tableau brut est conservée afin de vérifier en fin de notebook que la préparation ne l'a pas modifié.

In [3]:
bruts = {nom: charger_brut(nom) for nom in ["regional", "survey", "cropcut"]}
copies_brutes = {nom: df.copy() for nom, df in bruts.items()}
tableaux = preparer_tout(bruts)

regional = tableaux["regional_prepare"]
survey = tableaux["survey_prepare"]
cropcut = tableaux["cropcut_prepare"]


def bilan_indicateurs(df):
    colonnes = [colonne for colonne in df.columns if colonne.startswith("flag_")]
    return df[colonnes].sum().rename("lignes_signalees").to_frame()

### 3. Préparation de `regional`

Les corrections appliquées sont, dans l'ordre, la normalisation des textes, le remplacement des codes -999 et -99, la permutation de la surface et de la production sur les lignes inversées, l'unification des codes du Ghana et des libellés de pays, l'harmonisation de la saison principale éthiopienne, la reconstruction du niveau administratif, puis la sélection d'une ligne par cellule.

In [4]:
brut = bruts["regional"]
print("Lignes avant", len(brut), "et après", len(regional))
print("Rendements manquants avant", brut["yieldTonHa"].isna().sum(), "et après", regional["yieldTonHa"].isna().sum())
print("Rendements négatifs avant", (brut["yieldTonHa"] < 0).sum(), "et après", (regional["yieldTonHa"] < 0).sum())
print("Codes pays L0_GID avant", brut["L0_GID"].nunique(), "et après", regional["L0_GID"].nunique())
print("Libellés de pays avant", brut["country"].nunique(), "et après", regional["country"].nunique())
print("Niveaux manquants avant", brut["level"].isna().sum(), "et après", regional["level"].isna().sum())
display(bilan_indicateurs(regional))

Lignes avant 197535 et après 197535
Rendements manquants avant 34584 et après 34650
Rendements négatifs avant 66 et après 0
Codes pays L0_GID avant 71 et après 47
Libellés de pays avant 49 et après 47
Niveaux manquants avant 491 et après 6


,lignes_signalees
flag_surface_production_permutees,33740
flag_relation_rendement_inexpliquee,1710
flag_niveau_ambigu,6
flag_gid_incomplet,453
flag_unite_hors_gadm,491
flag_hierarchie_gid_incoherente,488
flag_rendement_hors_bornes,4658
flag_doublon_cellule,0


In [5]:
print("Lignes écartées comme doublons de cellule, par source :")
print(regional.loc[regional["flag_doublon_cellule"], "Source"].value_counts().to_dict())
print("\nRendements hors bornes, par source :")
print(regional.loc[regional["flag_rendement_hors_bornes"], "Source"].value_counts().to_dict())

modelisation = tableaux["regional_modelisation"]
print("\nJeu de modélisation :", len(modelisation), "lignes")
print("Niveaux :", modelisation["level"].value_counts().sort_index().to_dict())
pays_absents = sorted(set(regional["country"]) - set(modelisation["country"]))
print("Pays présents uniquement au niveau national :", pays_absents)
print("Pays :", modelisation["L0_GID"].nunique(), "| cultures :", modelisation["crop"].nunique(),
      "| années :", modelisation["harvest_year"].min(), "-", modelisation["harvest_year"].max())
print("Sources :", modelisation["Source"].value_counts().to_dict())

Lignes écartées comme doublons de cellule, par source :
{}

Rendements hors bornes, par source :
{'HarvestStat': 1901, 'FAOSTAT': 1784, 'FAO_subnational': 383, 'RCeA': 357, 'AgroMAPS': 231, 'USAID_KadunaState': 2}

Jeu de modélisation : 145877 lignes
Niveaux : {np.int64(1): 57406, np.int64(2): 88471}


Pays présents uniquement au niveau national : ['Djibouti', 'Equatorial Guinea']
Pays : 45 | cultures : 33 | années : 1960 - 2023


Sources : {'HarvestStat': 118333, 'RCeA': 10715, 'AgroMAPS': 9958, 'FAO_subnational': 6799, 'USAID_KadunaState': 72}


**Constat**

- Les 66 rendements négatifs ont disparu. Les rendements manquants passent de 34 584 à 34 650, les 36 codes -99 et les 30 valeurs négatives dérivées du code -999 étant désormais traités comme manquants.
- Les 33 740 lignes inversées ont été permutées. Les 1 710 lignes qui ne vérifient aucune relation entre production, surface et rendement restent signalées.
- Les 71 codes `L0_GID` deviennent 47 et les 49 libellés de pays deviennent 47, chaque code ne portant plus qu'un nom. Le niveau administratif est reconstruit pour 485 des 491 lignes, les 6 lignes ambiguës restant signalées.
- Les 491 lignes sans `GDAM_ID` proviennent toutes de `HarvestStat` et nomment une unité de niveau 2 absente de GADM. Il s'agit de 18 unités créées par des redécoupages administratifs, comme les départements nigériens de 2011, les zones éthiopiennes Buno Bedele et Southeast Tigray ou trois districts ougandais. Chacune porte l'identifiant de l'unité qui la contenait, et 423 lignes ne portent que celui de leur région. Leur niveau reconstruit vaut donc 2, et elles sont signalées par `flag_unite_hors_gadm`.
- Les départements de Tibiri, Falmey et Dioudiou, situés dans la région de Dosso, portent l'identifiant de la région de Diffa, soit 90 lignes.
- Une fois les unités hors GADM tenues à l'écart de la sélection, la règle d'une ligne par cellule n'écarte plus aucune ligne.
- 4 658 rendements sortent des bornes de plausibilité de leur culture, dont 1 784 lignes `FAOSTAT` de niveau national. Ce total inclut l'unique rendement de la culture `Legumes`, nul et donc dépourvu de bornes, qui est écarté par la règle selon laquelle un rendement nul ou négatif n'est jamais plausible.
- Le jeu de modélisation compte 145 877 lignes réparties sur 45 pays, 33 cultures et la période 1960 à 2023. Djibouti et la Guinée équatoriale en sont absents, car ils ne disposent que de données nationales.

**Analyse**

Une unité hors GADM partage la zone de son unité mère et ne peut pas recevoir de contour. Lorsqu'elle participait à la sélection par cellule, elle écartait les rendements de l'unité mère. La version précédente de la préparation écartait ainsi 182 lignes, dont 105 rendements régionaux `FAO_subnational` d'Oromia et du Tigray remplacés par le rendement d'une seule zone, ce qui avait été attribué à tort à un recouvrement entre sources. Le département d'Aderbissinat, issu de celui de Tchirozérine, reprend d'ailleurs le rendement de ce dernier dans 49 des 65 cellules communes. Exclure ces unités retire 383 lignes du jeu de modélisation et y restaure 110 rendements.

La hiérarchie GADM reste incohérente pour 488 lignes ghanéennes, dont l'identifiant de niveau 2 est rattaché à plusieurs régions. Seuls les polygones GADM permettraient de les corriger. `HarvestStat` représente 81 % du jeu de modélisation, ce qui en fait la source dominante de la cible.

**Conséquences pour la suite**

- Les lignes ghanéennes signalées par `flag_hierarchie_gid_incoherente` devront être exclues de toute agrégation au niveau 1 tant que leur rattachement n'est pas corrigé.
- La source devra être prise en compte dans l'évaluation, compte tenu de la prépondérance de `HarvestStat`.
- Les unités hors GADM ne pourront être réintégrées qu'avec des contours plus récents que ceux de GADM.

### 4. Préparation de `survey`

Les corrections appliquées sont la normalisation des textes, la suppression des copies exactes, l'harmonisation du libellé de la Côte d'Ivoire et de l'année de récolte, le signalement des valeurs impossibles et des surfaces GPS recopiées, et la construction des identifiants de ligne, de ménage et de grappe.

In [6]:
brut = bruts["survey"]
print("Lignes avant", len(brut), "et après", len(survey))
print("Libellé de pays remplacé", sorted(set(brut["Country"]) - set(survey["Country"])), "par", sorted(set(survey["Country"]) - set(brut["Country"])))
print("AgYearEnd manquant avant", brut["AgYearEnd"].isna().sum(), "et après", survey["AgYearEnd"].isna().sum())
display(bilan_indicateurs(survey))

modelisation = tableaux["survey_modelisation"]
print("Jeu de modélisation :", len(modelisation), "lignes")
print("Identifiant de ménage manquant :", modelisation["id_menage"].isna().sum(),
      "| identifiant de grappe manquant :", modelisation["id_grappe"].isna().sum())
print("\nLignes par pays, rendement fondé sur la surface déclarée et sur la surface GPS :")
display(survey.groupby("Country")[["Yield_ton_ha_", "YieldUsingGPS_ton_ha_"]].count())
print("Pays :", modelisation["Country"].nunique(), "| cultures :", modelisation["Crop"].nunique(),
      "| années :", modelisation["annee_recolte"].min(), "-", modelisation["annee_recolte"].max())

Lignes avant 195152 et après 194704


Libellé de pays remplacé ['Cote dIvoire'] par ['Côte dIvoire']
AgYearEnd manquant avant 218 et après 0


,lignes_signalees
flag_prccrop_invalide,80
flag_surface_aberrante,1
flag_surface_gps_recopiee,7403
flag_gps_nul,518
flag_hhid_tronque,3416
flag_hierarchie_gid_incoherente,10
flag_rendement_hors_bornes,737


Jeu de modélisation : 151138 lignes
Identifiant de ménage manquant : 6106 | identifiant de grappe manquant : 13558

Lignes par pays, rendement fondé sur la surface déclarée et sur la surface GPS :


,Yield_ton_ha_,YieldUsingGPS_ton_ha_
Country,,
Benin,3834,69
Burkina Faso,1570,0
Côte dIvoire,1333,609
Ethiopia,18682,27504
Guinea-Bissau,1368,1071
Malawi,49137,48352
Mali,4037,5643
Niger,8862,13561
Nigeria,4638,11545


Pays : 12 | cultures : 25 | années : 2005 - 2022


In [7]:
valides = survey[~(survey["flag_prccrop_invalide"] | survey["flag_surface_aberrante"])]
communes = valides[(valides["Yield_ton_ha_"] > 0) & (valides["YieldUsingGPS_ton_ha_"] > 0)]
ecart_log = np.log(communes["Yield_ton_ha_"] / communes["YieldUsingGPS_ton_ha_"])
ecart_fort = ecart_log.abs() > np.log(1.5)

par_pays = communes.assign(ecart_log=ecart_log, ecart_fort=ecart_fort).groupby("Country")
comparaison = pd.DataFrame({
    "parcelles": par_pays.size(),
    "rapport_median": np.exp(par_pays["ecart_log"].median()),
    "part_ecarts_superieurs_50_pct": par_pays["ecart_fort"].mean(),
}).round(3)

print("Parcelles disposant des deux rendements :", len(communes))
print("Rapport médian entre rendement déclaré et rendement GPS :", round(float(np.exp(ecart_log.median())), 3))
print("Part des parcelles dont les deux rendements diffèrent de plus de 50 % :", round(float(ecart_fort.mean()), 3))
display(comparaison)

couverture_regionale = tableaux["regional_modelisation"].groupby("country")["level"].value_counts().unstack(fill_value=0)
print("Couverture de regional_modelisation pour les pays sans cible GPS :")
display(couverture_regionale.loc[["Burkina Faso", "Benin", "Côte dIvoire"]])

recopie = tableaux["survey_modelisation"].groupby("Source")["flag_surface_gps_recopiee"].agg(["size", "mean"]).round(3)
recopie.columns = ["parcelles", "part_surface_gps_recopiee"]
print("Vagues dont plus de 10 % des surfaces GPS égalent strictement la surface déclarée :")
display(recopie[recopie["part_surface_gps_recopiee"] > 0.10])

Parcelles disposant des deux rendements : 117719
Rapport médian entre rendement déclaré et rendement GPS : 0.994
Part des parcelles dont les deux rendements diffèrent de plus de 50 % : 0.438


,parcelles,rapport_median,part_ecarts_superieurs_50_pct
Country,,,
Benin,50,1.002,0.320
Côte dIvoire,525,1.001,0.364
Ethiopia,17756,1.126,0.465
Guinea-Bissau,622,1.000,0.691
Malawi,44921,0.920,0.507
Mali,1869,1.000,0.317
Niger,6855,0.907,0.474
Nigeria,3825,0.742,0.562
Senegal,1327,1.000,0.398


Couverture de regional_modelisation pour les pays sans cible GPS :


level,1,2
country,,
Burkina Faso,2404,12923
Benin,0,19751
Côte dIvoire,80,0


Vagues dont plus de 10 % des surfaces GPS égalent strictement la surface déclarée :


,parcelles,part_surface_gps_recopiee
Source,,
LSMS_MLI2018,1853,0.151
LSMS_SEN2018,418,0.115
LSMS_TGO2021,8167,0.436
LSMS_UGA2019,854,0.160


**Constat**

- Les 448 copies exactes sont supprimées et le fichier passe de 195 152 à 194 704 lignes. L'année de fin de campagne est désormais renseignée pour toutes les lignes.
- 3 416 identifiants de ménage tronqués sont signalés. L'identifiant de ménage est manquant pour 6 106 lignes du jeu de modélisation et l'identifiant de grappe pour 13 558 lignes, faute de coordonnées. Ce total comprend 135 lignes dont la latitude et la longitude valent zéro, un point situé en mer qui code une position absente. Ces coordonnées sont remplacées par des valeurs manquantes et signalées par `flag_gps_nul`, ce qui concerne 518 lignes du fichier préparé.
- Le jeu de modélisation compte 151 138 lignes, 25 cultures et la période 2005 à 2022, mais seulement 12 pays sur 13.
- La cible fondée sur la surface GPS est absente pour l'ensemble du Burkina Faso et quasi absente pour le Bénin, avec 69 lignes contre 3 834 pour la cible fondée sur la surface déclarée. Elle est également réduite de moitié pour la Côte d'Ivoire.
- Sur les 117 719 parcelles qui disposent des deux rendements, le rapport médian entre le rendement fondé sur la surface déclarée et le rendement GPS vaut 0,994. Les deux valeurs diffèrent pourtant de plus de 50 % pour 43,8 % des parcelles. Cette part varie de 22,9 % au Togo à 69,1 % en Guinée-Bissau, et le rapport médian descend à 0,742 au Nigeria.
- 7 403 surfaces GPS sont strictement égales à la surface déclarée. Cette égalité concerne 43,6 % des parcelles de la vague `LSMS_TGO2021` du jeu de modélisation, et entre 11,5 et 16 % des parcelles de `LSMS_MLI2018`, `LSMS_SEN2018` et `LSMS_UGA2019`.
- Le Burkina Faso et le Bénin restent couverts par `regional_modelisation`, avec respectivement 15 327 et 19 751 lignes.

**Analyse**

Les enquêtes du Burkina Faso et, en grande partie, celles du Bénin n'ont pas mesuré la surface par GPS. Ce point n'avait pas été mis en évidence pays par pays lors du Data Understanding.

Recourir au rendement fondé sur la surface déclarée lorsque la surface GPS manque introduirait un bruit de mesure considérable dans la cible. Le rapport médian proche de 1 masque un désaccord fréquent, puisque les deux méthodes divergent de plus de moitié pour près d'une parcelle sur deux. Ce bruit serait en outre entièrement confondu avec le pays dans le cas du Burkina Faso, qui ne dispose que de la surface déclarée. Un indicateur de méthode ne permettrait donc pas de séparer l'effet de la méthode de mesure de celui du pays. La surface déclarée présente enfin un arrondi au quart d'acre et un écart qui dépend de la taille de la parcelle, établis lors du Data Understanding.

Une égalité stricte entre deux mesures indépendantes est improbable. Sa fréquence dans `LSMS_TGO2021` suggère que la surface déclarée a été reprise lorsque la mesure GPS manquait. Cette hypothèse reste à confirmer dans la documentation de l'enquête.

**Décisions**

- La cible `YieldUsingGPS_ton_ha_` est maintenue, sans remplacement par le rendement fondé sur la surface déclarée. Ce choix a été arrêté le 3 novembre 2026 sur délégation du porteur du projet.
- `Yield_ton_ha_` reste disponible dans `survey_prepare` pour des analyses de sensibilité.
- Les surfaces GPS recopiées sont signalées par `flag_surface_gps_recopiee` sans être écartées.

**Conséquences pour la suite**

- Le modèle à l'échelle de la parcelle couvre 12 pays. Son application au Burkina Faso relèverait d'une extrapolation hors du domaine d'entraînement, ce pays étant couvert par la seule cible principale.
- L'évaluation devra mesurer la performance avec et sans les surfaces GPS recopiées, en particulier pour le Togo.
- Lorsque l'identifiant de ménage est manquant, les partitions devront s'appuyer sur la grappe ou, à défaut, sur `L2_GID`.

### 5. Préparation de `cropcut`

Les corrections appliquées sont la normalisation des textes, la reconstruction du ménage par division entière de `holder_id` par 100, le signalement des poids incohérents et des dates impossibles dans le calendrier éthiopien, et la construction de l'année de récolte et de la grappe.

In [8]:
display(bilan_indicateurs(cropcut))
validation = tableaux["cropcut_validation"]
print("Jeu de validation :", len(validation), "lignes")
print("Ménages reconstruits :", cropcut["id_menage"].nunique(), "| household_id d'origine :", bruts["cropcut"]["household_id"].nunique())
print("Identifiant de ménage manquant, par vague :", cropcut.loc[cropcut["id_menage"].isna(), "Source"].value_counts().to_dict())

,lignes_signalees
flag_poids_incoherent,80
flag_date_mesure_invalide,19
flag_rendement_hors_bornes,2


Jeu de validation : 11263 lignes
Ménages reconstruits : 2338 | household_id d'origine : 2334
Identifiant de ménage manquant, par vague : {'ETH2018': 2905}


**Constat**

- 80 mesures présentent un poids sec supérieur au poids frais et 19 dates de mesure sont impossibles, même dans le calendrier éthiopien. Seuls 2 rendements sortent des bornes de plausibilité.
- La reconstruction produit 2 338 ménages, contre 2 334 valeurs d'origine de `household_id`. Les deux valeurs de substitution de `ETH2011` sont remplacées par les ménages réels qu'elles masquaient.
- L'identifiant de ménage reste manquant pour les 2 905 lignes de `ETH2018`, vague qui ne fournit pas `holder_id`.
- Le jeu de validation compte 11 263 lignes.

**Conséquences pour la suite**

Les dates invalides n'affectent pas la cible et sont seulement signalées. Les mesures de `ETH2018` devront être regroupées par grappe pour toute validation par groupe.

### 6. Contrôles d'intégrité

Ces contrôles vérifient les propriétés attendues des jeux produits. Le notebook s'interrompt si l'un d'eux échoue.

In [9]:
controles = {
    "aucun rendement négatif dans regional": (regional["yieldTonHa"].dropna() >= 0).all(),
    "un seul nom de pays par L0_GID": regional.groupby("L0_GID")["country"].nunique().max() == 1,
    "une ligne retenue par cellule dans regional": not tableaux["regional_modelisation"].duplicated(
        subset=["GDAM_ID", "L1_GID", "L2_GID", "crop", "saison", "harvest_year"]).any(),
    "identifiant de ligne unique dans survey": survey["id_ligne"].is_unique,
    "aucune copie exacte dans survey": not survey.drop(columns=[c for c in survey.columns if c.startswith(("flag_", "id_", "annee_"))]).duplicated().any(),
    "cibles renseignées dans les jeux finaux": all(
        tableaux[nom][cible].notna().all()
        for nom, cible in [("regional_modelisation", "yieldTonHa"), ("survey_modelisation", "YieldUsingGPS_ton_ha_"), ("cropcut_validation", "Yield_ton_ha_")]
    ),
    "données brutes non modifiées par la préparation": all(bruts[nom].equals(copies_brutes[nom]) for nom in bruts),
}
for nom, resultat in controles.items():
    print(f"{'OK ' if resultat else 'ÉCHEC'} {nom}")
assert all(controles.values())

OK  aucun rendement négatif dans regional
OK  un seul nom de pays par L0_GID
OK  une ligne retenue par cellule dans regional
OK  identifiant de ligne unique dans survey
OK  aucune copie exacte dans survey
OK  cibles renseignées dans les jeux finaux
OK  données brutes non modifiées par la préparation


### 7. Export des jeux préparés

Les tableaux sont écrits au format CSV compressé dans `data/processed`. Ce dossier est exclu du dépôt git, les fichiers étant entièrement reproductibles à partir des données brutes et du code.

In [10]:
exporter(tableaux)
for nom, tableau in tableaux.items():
    chemin = DOSSIER_TRAITE / f"{nom}.csv.gz"
    print(f"{nom:<24} {tableau.shape[0]:>7} lignes  {tableau.shape[1]:>3} colonnes  {chemin.stat().st_size / 1e6:6.2f} Mo")

bornes_plausibilite           34 lignes    2 colonnes    0.00 Mo
regional_prepare          197535 lignes   34 colonnes    4.05 Mo
regional_modelisation     145877 lignes   34 colonnes    3.44 Mo
survey_prepare            194704 lignes   38 colonnes    7.57 Mo
survey_modelisation       151138 lignes   38 colonnes    6.26 Mo
cropcut_prepare            11345 lignes   26 colonnes    0.26 Mo
cropcut_validation         11263 lignes   26 colonnes    0.25 Mo


### 8. Synthèse

| Jeu produit | Lignes | Usage |
|---|---:|---|
| `regional_prepare` | 197 535 | Fichier regional corrigé, avec tous ses indicateurs |
| `regional_modelisation` | 145 877 | Cible principale aux niveaux 1 et 2, une ligne par cellule |
| `survey_prepare` | 194 704 | Fichier survey corrigé, avec tous ses indicateurs |
| `survey_modelisation` | 151 138 | Cible à l'échelle de la parcelle |
| `cropcut_prepare` | 11 345 | Fichier cropcut corrigé, avec tous ses indicateurs |
| `cropcut_validation` | 11 263 | Référence de validation pour l'Éthiopie |
| `bornes_plausibilite` | 34 | Bornes du filtre des auteurs par culture |

**Décision**

Les deux premières conditions de la décision issue du Data Understanding sont remplies. Les fichiers sont traités séparément et les corrections sont implémentées sous forme de fonctions testées. La troisième condition, la définition du protocole d'évaluation par groupe et par période, reste à remplir avant l'entraînement du premier modèle.

**Choix de la cible de `survey`**

La cible `YieldUsingGPS_ton_ha_` est maintenue après comparaison avec le rendement fondé sur la surface déclarée. Le mélange des deux méthodes introduirait un bruit de mesure confondu avec le pays. Le Burkina Faso et le Bénin restent couverts par la cible principale de `regional`.